# Chọn và thiết kế đặc trưng

Dùng dữ liệu giá nhà đã làm sạch (các bước như notebook 04) để minh họa: tạo đặc trưng mới, đánh giá mức độ liên quan, loại đặc trưng không cần thiết, giảm chiều bằng PCA.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path('..') / 'data'
df = pd.read_csv(DATA / 'house_prices_raw.csv').drop_duplicates()   # làm sạch như notebook 04
df['district'] = df['district'].str.strip().str.lower()
df = df.dropna(subset=['price'])
df = df[df['area'] <= 286.55]                                       # ngưỡng IQR tính ở notebook 04
print(df.shape)

(989, 7)


## 1. Thiết kế đặc trưng mới

Năm xây dựng chưa phải dạng tốt nhất để diễn giải: điều ảnh hưởng đến giá là **tuổi nhà**. Tương tự, có thể tạo diện tích trung bình mỗi phòng.

In [2]:
df['age'] = 2026 - df['year_built']                    # tuổi nhà (năm)
df['area_per_bedroom'] = df['area'] / df['bedrooms']   # diện tích trung bình mỗi phòng ngủ
df[['year_built', 'age', 'area', 'bedrooms', 'area_per_bedroom']].head()

,year_built,age,area,bedrooms,area_per_bedroom
0,2024.0,2.0,173.2,4.0,43.300000
1,2011.0,15.0,123.5,4.0,30.875000
2,NaN,NaN,129.8,3.0,43.266667
3,1995.0,31.0,69.2,1.0,69.200000
4,2011.0,15.0,110.6,3.0,36.866667


## 2. Đánh giá mức độ liên quan của đặc trưng số: hệ số tương quan

In [3]:
numeric = ['area', 'bedrooms', 'distance', 'age', 'area_per_bedroom']
df[numeric + ['price']].corr()['price'].round(3).sort_values()

distance           -0.226
area_per_bedroom   -0.087
age                -0.062
bedrooms            0.841
area                0.944
price               1.000
Name: price, dtype: float64

## 3. Đặc trưng phân loại: so sánh giá trung bình theo nhóm

In [4]:
print(df.groupby('district')['price'].agg(['count', 'mean']).round(0))
print(df.groupby('paint_color')['price'].agg(['count', 'mean']).round(0))

           count    mean
district                
center       183  4475.0
outskirts    389  3395.0
suburb       417  3921.0
             count    mean
paint_color               
blue           269  3833.0
gray           246  3945.0
white          238  3790.0
yellow         236  3691.0


**Nhận xét:** giá trung bình chênh lệch rõ giữa các khu vực (trung tâm đắt hơn ngoại thành hơn 1 tỷ), nhưng gần như không đổi theo màu sơn (chênh lệch nhỏ, nằm trong mức dao động ngẫu nhiên). Ta **giữ** `district` (mã hóa one-hot) và **loại** `paint_color`, đúng như ví dụ trong giáo trình Chương 2.

## 4. Đặc trưng tương quan mạnh và giảm chiều bằng PCA

Diện tích và số phòng ngủ tương quan mạnh với nhau. PCA tìm một trục mới $z = a_1 x_1 + a_2 x_2$ sao cho phương sai của dữ liệu theo trục đó lớn nhất (giáo trình Chương 2).

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pair = df[['area', 'bedrooms']].dropna()
print('Hệ số tương quan diện tích – số phòng:', round(pair.corr().iloc[0, 1], 3))

pair_scaled = StandardScaler().fit_transform(pair)    # chuẩn hóa trước khi dùng PCA
pca = PCA(n_components=2).fit(pair_scaled)
print('Tỉ lệ phương sai giữ lại theo từng trục:', pca.explained_variance_ratio_.round(3))
print('Trục thứ nhất z = a1*x1 + a2*x2 với (a1, a2) =', pca.components_[0].round(3))

Hệ số tương quan diện tích – số phòng: 0.86
Tỉ lệ phương sai giữ lại theo từng trục: [0.93 0.07]
Trục thứ nhất z = a1*x1 + a2*x2 với (a1, a2) = [0.707 0.707]


Trục thứ nhất giữ phần lớn phương sai của hai đặc trưng, nên có thể thay hai cột bằng một cột $z$ mà mất ít thông tin. Với bài toán giá nhà chỉ có vài đặc trưng, ta vẫn giữ nguyên hai cột để dễ diễn giải; PCA hữu ích hơn khi dữ liệu có hàng trăm, hàng nghìn chiều.